# Chapter 1 — RAG를 위한 Document Parser

> Part I · Rule-based · VLM · LLM Reasoning · Structured Output · Tree Index
> v2.0 / 2026 · NOWAVE

## 튜토리얼 구성 (6개)

| § | 튜토리얼 | 도구 | 결과물 |
|---|---|---|---|
| 1-1 | 베이스라인 (규칙 기반) | PyMuPDF4LLM | Markdown |
| 1-2 | 30+ 포맷 통합 (규칙 기반) | Unstructured | Element 객체 |
| 1-3 | 표 보존 (VLM 기반) | Docling | Markdown + Table objects |
| 1-4 | 클라우드 파싱 (VLM 기반) | LlamaParse | Markdown (한국어 지원) |
| 1-5 | **Structured Output** | GPT-4o + Pydantic + instructor | typed JSON |
| 1-6 | **Tree Index Builder** | PyMuPDF + GPT-4o-mini | PageIndex 호환 JSON |

> §1-5 산출물 → Ch.3·Ch.5의 typed extraction 입력
> §1-6 산출물 → Ch.3·Ch.4·Ch.5·Ch.6의 retrieval 인덱스

## 0. 환경 준비

OpenAI API 키와 LlamaCloud API 키(선택)를 환경 변수로 설정한 뒤 실행한다. 본 노트북의 외부 API 호출 총 비용은 약 $2~$5다.

**필요 패키지**: `pymupdf`, `pymupdf4llm`, `unstructured[pdf]`, `docling`, `llama-parse`, `openai`, `pydantic`, `instructor`, `python-dotenv`, `reportlab` (샘플 PDF 생성용)

In [9]:
#%pip install transformers -U

In [1]:
#%pip install -q pymupdf pymupdf4llm unstructured[pdf], docling openai pydantic instructor python-dotenv reportlab

### 0.1 환경 변수 로드

`.env` 파일에 API 키를 저장한 뒤 `load_dotenv()`로 읽어들인다. `OPENAI_API_KEY`는 §1-5와 §1-6의 LLM 호출에 필수다.

In [1]:
# ─────────────────────────────────────────────────
# 환경 변수 로드 — .env 파일에서 OPENAI_API_KEY 읽기
# ─────────────────────────────────────────────────
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# .env 파일을 자동으로 찾아 환경 변수에 로드
load_dotenv()

# 키가 없으면 sk-... placeholder로 대체 (실제 호출 시 에러)
# os.environ.setdefault("OPENAI_API_KEY", "sk-...")
# os.environ.setdefault("LLAMA_CLOUD_API_KEY", "llx-...")  # §1-4 선택 사용

# 작업 디렉터리 설정 — 모든 챕터가 ./work/ 하위에 산출물 저장·읽기
WORK = Path("./work")
WORK.mkdir(exist_ok=True)
print("작업 디렉터리:", WORK.resolve())

작업 디렉터리: /Users/namuai/06-claude/RAG/work


### 0.2 샘플 PDF 생성 — 재현 가능한 합성 SEC 10-K

실제 SEC 필링을 다운로드하는 대신, ReportLab으로 **표·헤딩이 포함된 합성 10-K 보고서**를 자체 생성한다. 이렇게 하면:
- 네트워크 없이 재현 가능
- 모든 챕터에서 동일한 입력 데이터 보장
- 표 추출 정확도 측정용 ground truth 제공

In [2]:
# ─────────────────────────────────────────────────────────
# 샘플 10-K PDF 생성 — ReportLab으로 표·헤딩 포함 PDF 작성
# ─────────────────────────────────────────────────────────
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle,
    Paragraph, Spacer, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet

styles = getSampleStyleSheet()
pdf_path = WORK / "sample_10k.pdf"

# SimpleDocTemplate으로 다중 페이지 PDF 작성
doc = SimpleDocTemplate(str(pdf_path), pagesize=LETTER)
story = []

# 표지
story.append(Paragraph("<b>FY2025 Annual Report — Sample Corp</b>", styles["Title"]))
story.append(Spacer(1, 18))

# Chapter 1 — Revenue (매출)
story.append(Paragraph("<b>Chapter 1. Revenue</b>", styles["Heading1"]))
story.append(Paragraph("<b>1.1 Q1 Results</b>", styles["Heading2"]))

# 표 — 분기별 사업부문 매출 (Ch.3·5의 typed extraction 대상)
data = [["Segment", "Q1 Revenue (M$)", "YoY %"],
        ["iPhone", "69,702", "+5.5%"],
        ["Mac",    "7,744",  "+1.6%"],
        ["Services","26,375","+11.5%"]]
t = Table(data, colWidths=[140, 130, 100])
t.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2C3E50")),
    ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
    ("GRID",       (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
]))
story.append(t)
story.append(Spacer(1, 12))

# Chapter 2 — Risk Factors (리스크 요인)
story.append(Paragraph("<b>Chapter 2. Risk Factors</b>", styles["Heading1"]))
story.append(Paragraph(
    "Supply chain disruption is the primary risk identified.",
    styles["BodyText"]
))

doc.build(story)
print("샘플 PDF 생성 완료:", pdf_path, f"({pdf_path.stat().st_size:,} bytes)")
print()
print("이 파일은 본 책의 모든 챕터(Ch.2~Ch.7)에서 입력으로 재사용된다.")

샘플 PDF 생성 완료: work/sample_10k.pdf (2,064 bytes)

이 파일은 본 책의 모든 챕터(Ch.2~Ch.7)에서 입력으로 재사용된다.


---
## §1-1 PyMuPDF4LLM — 규칙 기반 베이스라인

**왜 베이스라인부터 시작하는가?**
- 비용 0원·속도 최고로 RAG 파이프라인의 출발점을 만든다.
- 다른 모든 파서가 이보다 우월한지 정량 비교의 기준이 된다.
- Chapter 4(VectorlessRAG 직접 구현)에서 raw 텍스트 추출의 표준 도구로 재사용한다.

**작동 원리**: PDF의 내부 객체 모델(텍스트 블록·좌표·폰트)을 읽어 마크다운으로 변환. 표는 `|` 구분의 마크다운 표 문법으로 출력된다.

**한계**: 셀 병합·다단 레이아웃·세로쓰기에서 헤더와 데이터가 어긋날 수 있다.

In [3]:
# ─────────────────────────────────────────────────────
# §1-1: PyMuPDF4LLM 베이스라인 — 마크다운 추출
# ─────────────────────────────────────────────────────
import pymupdf4llm
import time

# page_chunks=True → 페이지 단위로 분할된 리스트 반환
# (§1-6의 Tree Builder가 페이지 정보를 사용)
t0 = time.time()
md_pages = pymupdf4llm.to_markdown(
    str(pdf_path),
    page_chunks=True,        # [{"text": ..., "metadata": ...}, ...]
)
elapsed = time.time() - t0

print(f"처리 시간: {elapsed:.2f}초")
print(f"페이지 수: {len(md_pages)}")
print("\n---- 첫 페이지 미리보기 ----")
print(md_pages[0]["text"][:600])

=== Document parser messages ===
Using RapidOCR for OCR processing.

처리 시간: 0.62초
페이지 수: 1

---- 첫 페이지 미리보기 ----
## **FY2025 Annual Report — Sample Corp** 

## **Chapter 1. Revenue** 

## **1.1 Q1 Results** 

|**Segment**|**Q1 Revenue (M$)**|**YoY %**|
|---|---|---|
|iPhone|69,702|+5.5%|
|Mac|7,744|+1.6%|
|Services|26,375|+11.5%|



## **Chapter 2. Risk Factors** 

Supply chain disruption is the primary risk identified. 




**관찰 포인트**

표가 마크다운 표 문법으로 잘 추출되었는가? 셀 병합이 있다면 헤더 정렬이 흐트러질 수 있다. 이런 경우 §1-3 Docling이 효과적이다.

이 마크다운 출력은 Chapter 2(VectorRAG)에서 임베딩 입력으로 그대로 사용된다.

---
## §1-2 Unstructured — 30+ 포맷 통합 인터페이스

**왜 Unstructured인가?**
- PDF·HTML·DOCX·PPTX·EML 등 30+ 포맷을 동일한 `Element` 모델로 변환.
- 멀티 포맷 워크플로우(메일 첨부 + 위키 + PDF 보고서)의 표준 도구.
- `strategy` 옵션으로 정확도·속도를 즉시 조절.

**3가지 strategy**:
- `fast` — 규칙 기반, 가장 빠름 (본 데모에서 사용)
- `hi_res` — YOLOX·detectron2 레이아웃 모델, 정확하지만 GPU 권장
- `ocr_only` — 스캔본 PDF 전용

**Korean tip**: Unstructured는 한글을 NFD로 출력하는 경우가 있다. 후속 검색에서 매칭 실패의 원인이 되므로 NFC 정규화 단계를 반드시 거친다.

In [4]:
# ─────────────────────────────────────────────────────
# §1-2: Unstructured — partition_pdf로 element 추출
# ─────────────────────────────────────────────────────
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename=str(pdf_path),
    strategy="fast",                 # 시연용 — 프로덕션은 hi_res
    infer_table_structure=True,      # 표 셀 구조 인식
)

# Element 카테고리별 분포 — Title / NarrativeText / Table / ListItem 등
from collections import Counter
cats = Counter(e.category for e in elements)
print("Element 카테고리 분포:", dict(cats))

# 추출된 표 출력
tables = [e for e in elements if e.category == "Table"]
if tables:
    print("\n---- 첫 표 (text) ----")
    print(tables[0].text)
    # text_as_html은 hi_res 모드에서 더 풍부 — 셀 구조 보존
    if hasattr(tables[0].metadata, "text_as_html"):
        print("\n---- HTML ----")
        print(tables[0].metadata.text_as_html[:400])

/opt/miniconda3/envs/lecture/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Element 카테고리 분포: {'Title': 9, 'UncategorizedText': 6, 'NarrativeText': 1}


### NFC 정규화 — 한국어 안전장치

한국어 파싱에서 가장 흔한 함정은 NFD/NFC 불일치다. Unstructured 등 일부 파서는 한글 자모를 NFD로 분리해 출력하는데, 사용자 쿼리는 보통 NFC이므로 검색 시 매칭 실패가 발생한다.

In [5]:
# ─────────────────────────────────────────────────────
# 한글 NFC 정규화 — 후속 모든 단계에 적용 권장
# ─────────────────────────────────────────────────────
import unicodedata

def to_nfc(text: str) -> str:
    """한글 자모를 완성형(NFC)로 정규화"""
    return unicodedata.normalize("NFC", text)

# 모든 element의 text에 일괄 적용
normalized = [to_nfc(e.text) for e in elements if hasattr(e, "text")]
print(f"정규화된 element {len(normalized)}개")
print("\n팁: 한국어 PDF 처리 시 이 단계를 빼먹으면")
print("    Chapter 2의 임베딩 검색에서 매칭 실패가 발생한다.")

정규화된 element 16개

팁: 한국어 PDF 처리 시 이 단계를 빼먹으면
    Chapter 2의 임베딩 검색에서 매칭 실패가 발생한다.


---
## §1-3 Docling — VLM 기반 표 챔피언 (97.9% 정확도)

**왜 Docling인가?**
- IBM Research가 공개한 오픈소스 파서. 자체 레이아웃 모델 + TableFormer 표 인식기.
- 2026-05 기준 복합 표 추출 정확도 **97.9%**로 오픈소스 중 최고.
- 데이터 프라이버시 요구가 있는 도메인(금융·의료·공공)에서 셀프 호스팅 가능.

**핵심 API**:
- `DocumentConverter().convert(path)` — 파싱 실행
- `result.document.export_to_markdown()` — 마크다운 출력
- `result.document.tables` — 표 객체 리스트 (DataFrame 변환 가능)

In [8]:
# ─────────────────────────────────────────────────────
# §1-3: Docling — 표 추출 정확도 97.9%
# ─────────────────────────────────────────────────────
from docling.document_converter import DocumentConverter
import time

# Default Pipeline — Layout 인식 모델 + TableFormer 자동 포함
converter = DocumentConverter()

t0 = time.time()
docling_result = converter.convert(str(pdf_path))
elapsed = time.time() - t0
print(f"처리 시간: {elapsed:.2f}초")

# 마크다운 변환
md = docling_result.document.export_to_markdown()
print(f"마크다운 길이: {len(md):,}자")
print(f"표 객체 수: {len(docling_result.document.tables)}")

# 첫 표를 pandas DataFrame으로 변환
# → Chapter 5(3-Stage Architecture)의 Architect 단계 입력으로 사용 가능
if docling_result.document.tables:
    df = docling_result.document.tables[0].export_to_dataframe()
    print("\n---- 첫 표 (DataFrame) ----")
    print(df.head())

2026-05-18 18:31:50,545 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-05-18 18:31:50,548 - RapidOCR - INFO: File exists and is valid: /opt/miniconda3/envs/lecture/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
2026-05-18 18:31:50,548 - RapidOCR - INFO: Using /opt/miniconda3/envs/lecture/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
2026-05-18 18:31:50,567 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-05-18 18:31:50,568 - RapidOCR - INFO: File exists and is valid: /opt/miniconda3/envs/lecture/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
2026-05-18 18:31:50,568 - RapidOCR - INFO: Using /opt/miniconda3/envs/lecture/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
2026-05-18 18:31:50,581 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-05-18 18:31:50,586 - RapidOCR - INFO: File exists and is valid: /opt/miniconda3/envs/lecture/lib/python3.11/sit

처리 시간: 1.98초
마크다운 길이: 380자
표 객체 수: 1

---- 첫 표 (DataFrame) ----
          0                1       2
0   Segment  Q1 Revenue (M$)    YoY%
1    iPhone           69,702   +5.5%
2       Mac            7,744   +1.6%
3  Services           26,375  +11.5%


**핵심 포인트** — Docling이 반환한 마크다운은 그대로 §1-5 Structured Output 또는 §1-6 Tree Index의 입력으로 재사용 가능하다. 즉 **"Docling → GPT-4o" 2단계 파이프라인**이 정확도·비용 측면에서 가장 균형 잡힌 디폴트다.

이 패턴은 Chapter 7(최종 비교)의 통제 변수 표에서 모든 백엔드의 표준 파서로 채택된다.

---
## §1-4 Structured Output — GPT-4o + Pydantic + instructor

**왜 Structured Output이 중요한가?**
- LLM 추론 기반 파서의 첫 번째 핵심 용도.
- LLM이 PDF 페이지 이미지를 보고 **typed JSON**으로 정답을 반환한다.
- 결과가 곧 함수 호출 인수 — 후속 코드에서 `fs.segments[0].revenue`처럼 타입 보장된 필드 접근이 가능하다.

**Chapter 5(3-Stage Architecture)의 Stage 1 Architect 단계와 동일한 패턴이다.** 본 셀에서 만든 typed JSON은 Chapter 3·5에서 그대로 재사용한다.

**핵심 메커니즘**: OpenAI의 `response_format=PydanticModel`이 출력을 schema에 강제(strict)한다. Schema 위반은 0%다.

### 1-4-1. Pydantic 모델 정의 — 정답이 따라야 할 schema

In [9]:
# ─────────────────────────────────────────────────────
# §1-4: Pydantic 모델로 정답 schema 정의
# ─────────────────────────────────────────────────────
from pydantic import BaseModel, Field
from typing import Literal

class Segment(BaseModel):
    """사업부문 1개 — typed object"""
    name: str = Field(description="사업부문 명 (예: iPhone, Mac, Services)")
    revenue_m_usd: float = Field(description="매출액 (백만 달러)")
    yoy_growth_pct: float = Field(description="전년 동기 대비 성장률 (%)")

class FinancialStatement(BaseModel):
    """재무제표 1개 — 본 데모의 최종 산출물 schema"""
    fiscal_period: str = Field(description='회계 기간 예: "FY2025 Q1"')
    total_revenue_m_usd: float
    segments: list[Segment]

# schema를 JSON으로 확인 — LLM에게 강제될 형식
print("Pydantic 스키마:")
print(json.dumps(
    FinancialStatement.model_json_schema(),
    indent=2, ensure_ascii=False
)[:600])

Pydantic 스키마:
{
  "$defs": {
    "Segment": {
      "description": "사업부문 1개 — typed object",
      "properties": {
        "name": {
          "description": "사업부문 명 (예: iPhone, Mac, Services)",
          "title": "Name",
          "type": "string"
        },
        "revenue_m_usd": {
          "description": "매출액 (백만 달러)",
          "title": "Revenue M Usd",
          "type": "number"
        },
        "yoy_growth_pct": {
          "description": "전년 동기 대비 성장률 (%)",
          "title": "Yoy Growth Pct",
          "type": "number"
        }
      },
      "required": [
        "name",
        "revenue_m_us


### 1-5-2. PDF 페이지를 이미지로 렌더링

GPT-4o의 vision 입력은 PNG/JPG를 받는다. PyMuPDF의 `get_pixmap()`으로 페이지를 이미지로 변환한다.

In [10]:
# ─────────────────────────────────────────────────────
# PDF 첫 페이지를 PNG로 렌더링 (GPT-4o vision 입력용)
# ─────────────────────────────────────────────────────
import pymupdf

doc = pymupdf.open(str(pdf_path))
pix = doc[0].get_pixmap(dpi=180)        # dpi=180은 균형점 (높을수록 정확·비용↑)
img_path = WORK / "page1.png"
pix.save(str(img_path))
print(f"이미지 저장: {img_path}  ({img_path.stat().st_size:,} bytes)")

이미지 저장: work/page1.png  (68,556 bytes)


### 1-5-3. GPT-4o 호출 — `response_format`으로 schema 강제

이 셀이 본 챕터의 핵심이다. `beta.chat.completions.parse(response_format=Model)`로 출력을 Pydantic 모델에 강제하면:
- LLM은 schema를 위반하지 못한다
- 후속 코드에서 typed 필드 접근 보장
- 함수 호출·DB insert·API 응답으로 그대로 사용 가능

In [11]:
# ─────────────────────────────────────────────────────
# GPT-4o + Vision + Pydantic schema 강제 호출
# ─────────────────────────────────────────────────────
import base64
from openai import OpenAI

client = OpenAI()

# 이미지를 base64로 인코딩 — data URL 형태로 전달
with open(img_path, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

resp = client.beta.chat.completions.parse(
    model="gpt-5.4-mini",          # Structured Output 지원 모델
    messages=[{
        "role": "user",
        "content": [
            {"type": "text",
             "text": ("이 페이지의 사업부문별 매출 표를 JSON으로 정리하라. "
                      "수치는 백만 달러 단위로 통일한다.")},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
        ],
    }],
    response_format=FinancialStatement,   # ← typed 출력 강제 (핵심)
)

# parsed는 이미 Pydantic 인스턴스 — typed 필드 접근 보장
fs = resp.choices[0].message.parsed
print("회계 기간:", fs.fiscal_period)
print("총 매출:", fs.total_revenue_m_usd, "M$")
for s in fs.segments:
    print(f"  - {s.name:10s}  {s.revenue_m_usd:>10,.0f}M$  {s.yoy_growth_pct:+.1f}%")

회계 기간: FY2025 Q1
총 매출: 103821.0 M$
  - iPhone          69,702M$  +5.5%
  - Mac              7,744M$  +1.6%
  - Services        26,375M$  +11.5%


### 1-5-4. typed object → 함수 호출·DB insert로 즉시 사용

LLM의 결과가 더 이상 자연어 문자열이 아닌 **typed 객체**다. 이는 곧 후속 코드에서 함수 호출 인수·DB 컬럼 값으로 그대로 들어갈 수 있다는 뜻이다.

In [12]:
# ─────────────────────────────────────────────────────
# typed object 활용 시연 — 도메인 함수 직접 호출
# ─────────────────────────────────────────────────────

def query_segment_revenue(segments: list[Segment], target_name: str) -> float:
    """사업부문 이름으로 매출 조회 — typed 입력이라 타입 체커가 보장"""
    for s in segments:
        if target_name.lower() in s.name.lower():
            return s.revenue_m_usd
    return -1.0

iphone_rev = query_segment_revenue(fs.segments, "iPhone")
print(f"iPhone 매출 = {iphone_rev:,.0f}M$")

# JSON으로 저장 → Chapter 3·5의 typed extraction 입력으로 그대로 재사용
out_path = WORK / "structured_extraction.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(fs.model_dump(), f, ensure_ascii=False, indent=2)
print(f"\nChapter 3·5 입력 파일 저장: {out_path.resolve()}")

iPhone 매출 = 69,702M$

Chapter 3·5 입력 파일 저장: /Users/namuai/06-claude/RAG/work/structured_extraction.json


**대안 — `instructor` 라이브러리**: OpenAI 외에 Anthropic·Gemini 등에서도 동일 패턴을 1줄 데코레이터로 쓸 수 있다.

```python
import instructor
from openai import OpenAI
client = instructor.from_openai(OpenAI())
fs = client.chat.completions.create(
    model="gpt-5.4-mini",
    response_model=FinancialStatement,
    messages=[...],
)
```

---
## §1-6 Tree Index Builder — PageIndex 호환 구조 직접 만들기

**왜 트리 인덱스가 중요한가?**
- LLM 추론 기반 파서의 두 번째 핵심 용도.
- PDF를 청킹하지 않고 자연 계층(Chapter → Section)을 유지한 트리로 변환.
- **이 트리는 인덱스이자 retrieval 대상** — 벡터 DB 없이 LLM이 트리를 따라 추론·네비게이션.

**책 전체에서의 위치**: 본 셀에서 만든 `tree_index.json`은 Chapter 3·4·5·6에서 그대로 재사용된다.
- Chapter 3 — VectorlessRAG 이론·시각화
- Chapter 4 — pymupdf4llm + LangGraph로 직접 구현
- Chapter 5 — 3-Stage Architecture의 Stage 1 Architect 입력
- Chapter 6 — PageIndex SDK Cloud의 비교 baseline

**4단계 빌드**:
1. PyMuPDF로 raw 텍스트·페이지 추출 (규칙 기반)
2. GPT-4o-mini가 heading 계층 추론
3. 각 노드의 요약·키워드·page range 생성
4. JSON 직렬화

### 1-6-1. Stage 1 — Raw 텍스트·페이지 정보 추출

PyMuPDF가 담당한다. 모든 페이지의 텍스트를 페이지 번호와 함께 보관한다.

In [13]:
# ─────────────────────────────────────────────────────
# §1-6 Stage 1: 규칙 기반 raw 텍스트·페이지 추출
# ─────────────────────────────────────────────────────
import pymupdf

doc = pymupdf.open(str(pdf_path))
raw_pages = []
for i, page in enumerate(doc, start=1):
    raw_pages.append({
        "page": i,
        "text": page.get_text("text"),    # plain text 모드
    })

print(f"raw 페이지 수: {len(raw_pages)}")
print("페이지 1 (첫 200자):", raw_pages[0]["text"][:200])

raw 페이지 수: 1
페이지 1 (첫 200자): FY2025 Annual Report — Sample Corp
Chapter 1. Revenue
1.1 Q1 Results
Segment
Q1 Revenue (M$)
YoY %
iPhone
69,702
+5.5%
Mac
7,744
+1.6%
Services
26,375
+11.5%
Chapter 2. Risk Factors
Supply chain disru


### 1-6-2. Stage 2~3 — LLM이 heading 계층 + 노드 요약 생성

GPT-4o-mini가 한 번의 호출로 트리 전체를 생성한다. Pydantic schema로 출력 형식을 강제한다.

**왜 GPT-4o-mini인가?**
- 인덱싱은 비용에 민감 — 한 문서당 1회만 실행
- mini는 정확도/비용 비율이 가장 좋다
- gpt-4.1은 더 정확하지만 비용이 4~6배 비싸다

In [14]:
# ─────────────────────────────────────────────────────
# §1-6 Stage 2~3: LLM이 트리 노드를 한 번에 생성
# ─────────────────────────────────────────────────────
from pydantic import BaseModel
from typing import Optional

class TreeNode(BaseModel):
    """트리 노드 1개 — PageIndex 호환 형식"""
    node_id: str                              # 'n1', 'n1.1' 등
    title: str
    level: int                                 # 1=Chapter, 2=Section, ...
    page_start: int
    page_end: int
    summary: str                               # 50자 이내 (라우팅 메타데이터)
    keywords: list[str]                        # 핵심 키워드 3~5개
    children_ids: list[str] = []               # 부모-자식 관계

class TreeBuildResult(BaseModel):
    nodes: list[TreeNode]

# 모든 페이지 텍스트를 하나로 합치되 페이지 마커를 유지
# → LLM이 각 노드의 page_start/page_end를 추론하기 쉽도록
raw_text_concat = "\n\n".join(
    f"[PAGE {p['page']}]\n{p['text']}" for p in raw_pages
)

# GPT-5.4-mini로 트리 구축 (인덱싱은 1회만 — 캐시 권장)
resp = client.beta.chat.completions.parse(
    model="gpt-5.4-mini",                       # mini로 비용 절감
    messages=[{
        "role": "user",
        "content": (
            "다음 문서를 읽고 계층적 트리 인덱스를 구축하라. "
            "각 노드는 title, level(1=Chapter, 2=Section, 3=Subsection), "
            "page range, 한 줄 summary, 핵심 키워드 3~5개를 가진다. "
            "node_id는 'n1','n2'...로 부여하고 children_ids로 부모-자식 관계를 표현한다.\n\n"
            + raw_text_concat[:12000]          # 토큰 절감
        ),
    }],
    response_format=TreeBuildResult,           # ← schema 강제
)

tree = resp.choices[0].message.parsed
print(f"생성된 노드 수: {len(tree.nodes)}")
print()
print("트리 시각화 (depth 들여쓰기):")
for n in tree.nodes[:6]:
    indent = "  " * (n.level - 1)
    print(f"{indent}[{n.node_id}] L{n.level} {n.title}  (p.{n.page_start}-{n.page_end})")

생성된 노드 수: 3

트리 시각화 (depth 들여쓰기):
[n1] L1 Chapter 1. Revenue  (p.1-1)
  [n2] L2 1.1 Q1 Results  (p.1-1)
[n3] L1 Chapter 2. Risk Factors  (p.1-1)


### 1-6-3. Stage 4 — JSON 직렬화 + 비용 추정

생성된 트리를 JSON으로 저장한다. 이 파일이 후속 챕터들의 retrieval 인덱스가 된다.

In [15]:
# ─────────────────────────────────────────────────────
# §1-6 Stage 4: PageIndex 호환 JSON으로 직렬화
# ─────────────────────────────────────────────────────
tree_json_path = WORK / "tree_index.json"
with open(tree_json_path, "w", encoding="utf-8") as f:
    json.dump(tree.model_dump(), f, ensure_ascii=False, indent=2)
print("트리 인덱스 저장:", tree_json_path.resolve())
print(f"파일 크기: {tree_json_path.stat().st_size:,} bytes")

# 인덱싱 비용 추정 (대략치)
# GPT-4o-mini input: $0.15 / 1M tokens
input_tokens = len(raw_text_concat) // 4         # 매우 거친 추정 (1 token ≈ 4자)
cost_mini = input_tokens / 1_000_000 * 0.15
print(f"\n인덱싱 비용 추정: ~${cost_mini:.4f}  (입력 토큰 약 {input_tokens:,})")
print("\n이 비용은 일회성이다 — Ch.3·4·5·6에서 동일 트리를 재사용한다.")

트리 인덱스 저장: /Users/namuai/06-claude/RAG/work/tree_index.json
파일 크기: 1,163 bytes

인덱싱 비용 추정: ~$0.0000  (입력 토큰 약 61)

이 비용은 일회성이다 — Ch.3·4·5·6에서 동일 트리를 재사용한다.


### 1-6-4. LLM 트리 네비게이션 데모 — VectorlessRAG의 핵심

쿼리 시 LLM이 트리를 따라 정답 노드를 선택한다. 본 셀은 Chapter 3·4·5·6에서 본격적으로 다룰 패턴의 미니 버전이다.

In [16]:
# ─────────────────────────────────────────────────────
# 쿼리 시: LLM이 트리를 따라 추론·네비게이션 (VectorlessRAG 핵심)
# ─────────────────────────────────────────────────────

def llm_navigate(question: str, tree: TreeBuildResult, raw_pages: list[dict]):
    """트리 카탈로그를 LLM에게 보여주고 적합한 노드 1개 선택"""
    # 트리 전체를 한 페이지 카탈로그로 압축 — token 절감
    catalog = "\n".join(
        f"[{n.node_id}] L{n.level} {n.title} — {n.summary} "
        f"(p.{n.page_start}-{n.page_end})"
        for n in tree.nodes
    )
    msg = (
        f"다음은 문서의 트리 인덱스다.\n{catalog}\n\n"
        f"질문: {question}\n"
        "이 질문에 답하기 가장 적합한 노드의 node_id 하나만 출력하라."
    )
    r = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": msg}],
    )
    # 응답에서 첫 토큰만 추출 — '[n1]' 형식 처리
    chosen_id = r.choices[0].message.content.strip().split()[0].strip("[]")
    chosen = next((n for n in tree.nodes if n.node_id == chosen_id), None)
    if not chosen:
        return None

    # 선택된 노드의 페이지들로부터 본문 추출
    pages_text = "\n".join(
        p["text"] for p in raw_pages
        if chosen.page_start <= p["page"] <= chosen.page_end
    )
    return chosen, pages_text


# 실행
index_result = llm_navigate("iPhone Q1 매출이 얼마인가?", tree, raw_pages)
if index_result:
    node, ctx = index_result
    print(f"선택된 노드: [{node.node_id}] {node.title}")
    print(f"  page range: {node.page_start}-{node.page_end}")
    print(f"  요약: {node.summary}")

선택된 노드: [n2] 1.1 Q1 Results
  page range: 1-1
  요약: Q1 부문별 매출과 전년 대비 성장률이 표로 정리된다.


---
## §1-7 챕터 종합 — 6개 파서 결과 비교

이상의 6개 파서(파이프라인 5종 + 트리 빌더)를 동일 입력에 대해 돌렸을 때 보존된 정보량·산출물을 비교한다.

In [17]:
# ─────────────────────────────────────────────────────
# §1-7: 6개 파서의 결과 종합 비교
# ─────────────────────────────────────────────────────
import pandas as pd

comparison = pd.DataFrame([
    {"파서": "PyMuPDF4LLM",
     "패러다임": "규칙",
     "마크다운 길이": len(md_pages[0]["text"]),
     "표 검출": "△",
     "비용": "0",
     "산출물": "Markdown"},
    {"파서": "Unstructured",
     "패러다임": "규칙",
     "마크다운 길이": sum(len(e.text) for e in elements if hasattr(e,"text")),
     "표 검출": f"{len(tables)}개",
     "비용": "0",
     "산출물": "Elements"},
    {"파서": "Docling",
     "패러다임": "VLM",
     "마크다운 길이": len(md),
     "표 검출": f"{len(docling_result.document.tables)}개",
     "비용": "0",
     "산출물": "Markdown + Table"},
    {"파서": "GPT-5.4-mini (Structured)",
     "패러다임": "LLM 추론",
     "마크다운 길이": len(json.dumps(fs.model_dump())),
     "표 검출": f"{len(fs.segments)}개 segment",
     "비용": "~$0.02",
     "산출물": "Typed JSON"},
    {"파서": "GPT-5.4-mini (Tree)",
     "패러다임": "LLM 추론",
     "마크다운 길이": tree_json_path.stat().st_size,
     "표 검출": "—",
     "비용": f"~${cost_mini:.4f}",
     "산출물": "Tree JSON"},
])
comparison

,파서,패러다임,마크다운 길이,표 검출,비용,산출물
0,PyMuPDF4LLM,규칙,313,△,0,Markdown
1,Unstructured,규칙,222,0개,0,Elements
2,Docling,VLM,380,1개,0,Markdown + Table
3,GPT-5.4-mini (Structured),LLM 추론,283,3개 segment,~$0.02,Typed JSON
4,GPT-5.4-mini (Tree),LLM 추론,1163,—,~$0.0000,Tree JSON
